# 0 - Importando bibliotecas
---

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Definindo seed para resultados replicáveis
rng = np.random.default_rng(42)

# 1 - O algoritmo LMS
---
### 1.1 - Introdução 
Least Mean Squares (LMS) é uma algoritmo em que os pesos do modelo são ajustados para cada amostra ou grupo de amostras no sentido de diminuir um pouco o erro. No notebook da regressão linear o método para encontrar esses pesos era segundo o critério dos mínimos quadrados em que se chegava em um sistema de equações lineares cuja solução era diretamente os pesos que minimizavam o erro. Porém, neste notebook o algoritmo LMS visa também encontrar tais pesos mas de maneira iterativa utilizando porções do conjunto de pontos (partes do conjunto de treino) disponíveis para construir o modelo.

A partir dessa seção, e em notebooks seguintes, é muito conveniente utilizar uma linguagem mais específica para descrever o que está sendo feito e com quem:

1. Modelo: estrutura matemática que contém parâmetros, essa estrutura tem uma forma que consequentemente agrega um viés sobre o problema, por exemplo: linear, quadrático, polinomial, senoidal, 30 camadas com 5 neurônios em cada e etc.

2. Viés: Erro sistemático introduzido pelas hipóteses do modelo, por exemplo: assumir que o problema pode ser resolvido com uma reta (linear, modelo simples) quando no fundo o problema é mais complexo que isso e a escolha linear causa underfiting ou assumir que o problema pode ser resolvido por um polinomio de 1000 graus (polinomial, modelo complexo) quando no fundo o problema é menos complexo que isso e a escolha polinomial de grau alto causa overfiting.

3. Treinamento: Processo de cálculo dos parâmetros do modelo.

4. Hiperparamerâmetros: Não são os parâmetros calculados no treinamento mas são as características do modelo escolhidas previamente, por exemplo: Ao escolher um modelo polinomial um hiperparâmetro seria o grau do polinômio.

5. Dados: São os pontos, ou características (no inglês: features + labels), são números diretamente relacionados a uma ao problema, exemplos: 500.00 dólares, 26 graus celsius, 50 metros e etc, ou números diretamente relacionados a alguma descrição do problema: [1, 2, 3] => [perto, médio, longe] se $3$ então está longe, se $1$ então está perto, e se $2$?

7. Conjunto de treino: São os dados usados para o cálculo dos parâmetros do modelo no treinamento.

8. Conjunto de validação: São os dados usados para escolher o modelo ou os hiperparâmetros, com ele que se testa em conjunto com o treinamento se um modelo ou hiperparâmetros do modelo são adequados.

9. Conjunto de teste: Utilizado apenas após o treino, serve para fazer uma estimativa do desempenho do modelo em dados nunca antes vistos.

10. Iteração: Em algoritmos de aprendizado, cada iteração corresponde a um ciclo em que o algoritmo executa seus cálculos e, normalmente, atualiza seus parâmetros, no algoritmo LMS o conjunto de treino será dividido em partes e ao executar a sequência de cálculos do treinamento para uma parte se diz qua passa uma iteração assim que os pesos e viés é atualizado.

11. Época: Para o LMS, assim que todas as partes do conjunto de treino atualizaram os pesos e viés uma vez então é comtabilizado que passou uma época.

12. Mini Batch: nome associado a cada parte do conjunto de treino.

13. Batch: se refere ao conjunto de treino inteiro.

### 1.2 - Desenvolvendo a ideia do LMS
Partindo da regressão linear multivariada os dados são multidimencionais, $n$ dados com $m$ características: 

$$
(x_{11}, x_{12}, \ldots, x_{1m}, y_1),\;
(x_{21}, x_{22}, \ldots, x_{2m}, y_2),\;
\ldots,\;
(x_{n1}, x_{n2}, \ldots, x_{nm}, y_n)
$$

Equação geral:
$$
\boxed{
y = b + w_1x_1 + w_2x_2 + \cdots + w_mx_m \tag{1}
}
$$

Equação do erro:
$$
\boxed{
\begin{bmatrix}
e_1\\
e_2\\
\vdots\\
e_n
\end{bmatrix}
=
\begin{bmatrix}
y_1\\
y_2\\
\vdots\\
y_n
\end{bmatrix}
-
\begin{bmatrix}
1 & x_{11} & x_{12} & \cdots & x_{1m}\\
1 & x_{21} & x_{22} & \cdots & x_{2m}\\
\vdots & \vdots & \vdots & \ddots & \vdots\\
1 & x_{n1} & x_{n2} & \cdots & x_{nm}
\end{bmatrix}
\begin{bmatrix}
b\\
w_1\\
w_2\\
\vdots\\
w_m
\end{bmatrix}
\tag{2}
}
$$

A solução segundo o método dos mínimos quadrados já foi visto:
$$
X^TXW_0 = X^Td
$$

Portando basta penas resolver o sistema de equações para encontrar o $W_0$ que contém o viés $b_0$ e cada peso $w_{10}$, $w_{20}$, ..., $w_{m0}$.

Diferente dessa ideia, apresentada no notebook 1, o algoritmo LMS tem como objetivo ajustar de maneira iterativa o viés e os pesos para cada dado individual do conjunto de treino a fim de minimizar o erro segundo o Mean-Square Error (MSE), ou seja, para cada ponto do conjunto de treino é calculado um ajuste para viés e os pesos e se segue cada iteração até que o conjunto de treino todo seja varrido pelo algoritmo.

Para melhor vizualisar as contas, considerando a iteração i, lembrando que a cada iteração os pesos e viés são atualizados:

Na iteração i:
$$
d(i) =
y(i)
\qquad
x(i) =
\begin{bmatrix}
1\\ 
x_1(i)\\
x_2(i)\\
\vdots\\
x_m(i)
\end{bmatrix}
\qquad
W(i) =
\begin{bmatrix}
b(i)\\
w_1(i)\\
w_2(i)\\
\vdots\\
w_m(i)
\end{bmatrix}
\qquad
e(i) = d(i) - x^T(i)W(i-1)
$$

$d(i)$ e $x(i)$ são do dado do conjunto de treinamento que será utilizado para ajustar o peso e o viés na iteração $i$, $W(i)$ são os pesos e o viés pós ajuste da iteração $i$ e assim $e(i)$ é o erro calculado durante a iteração $i$ por isso utiliza os pesos da iteração anterior com $W(i-1)$, a iteração $i$ acaba após o ajuste dos pesos e viés.

MSE: 
$$
\boxed{
J_{MSE} = E\{e^2(i)\} \tag{3}
}
$$

**Observação** - Esperança matemática
$E\{...\}$: Esperança matemática, é uma medida que representa o valor médio esperado de uma variável aleatória, considerando todas as suas possíveis realizações e suas respectivas probabilidades. Em outras palavras, ela corresponde à média que seria obtida ao repetir um experimento aleatório um grande número de vezes.

$$
\mathbb{E}[X]
=
\sum_{i} x_i\,P(X=x_i)
$$

Para uma variável aleatória contínua, a esperança matemática é definida por:

$$
\mathbb{E}[X]
=
\int_{-\infty}^{\infty}
x\,f_X(x)\,dx
$$

Normalmente não se conhece a média real de uma váriavel aleatória, portanto utiliza-se uma estimativa da esperança matemática: 

$$
\mathbb{E}[X]
\approx
\frac{1}{N}\sum_{i=1}^{N}x_i
$$

De forma análoga, a esperança do erro quadrático pode ser estimada por:

$$
\boxed{
\mathbb{E}\left\{e^2(n)\right\}
\approx
\frac{1}{N}\sum_{i=1}^{N}e_i^2
} \tag{4}
$$
**Fim da observação**

Derivando o $J_{MSE}$ em relação a $W(i-1)$ e avaliando com o viés e pesos da iteração anterior o resultado é:
$$
\boxed{
\nabla_WJ_{MSE}(W(i-1)) = \frac{\partial E\{e^2(i)\}}{\partial W(i-1)} = -2E\{e(i)x(i)\} \tag{5}
}
$$
Que corresponde a um resultado muito familiar do notebook de regressão linear, e de fato ao igualar a equação 5 a zero o resultado é conhecido como solução de Wiener:

$$
E\{x(i)x^T(i)\}W^{Wiener} = E\{x(i)d(i)\}
$$

Essa solução não é iterativa mas a é ótima se estivesse disponível as esperanças matemáricas reais: $E\{x(i)x^T(i)\}$ e $E\{x(i)d(i)\}$, porém não estão restando duas abordagens: estimar essas duas esperanças a apartir da equação 4 utilizando dos os pontos do cnjunto de treino, ou utilizar um método iterativo para a apartir do gradiente (equação 5) ir decendo a curva da função do MSE (equação 3) por meio de 'passos' sucessivos até o mínimo do MSE, esses 'passos' correspondem a pequenos ajuste dos pesos e viéses da iteração anterior com o gradiente calculado na itereção atual:

$$
\boxed{
W(i) = W(i-1) - \frac{\eta}{2}\nabla_WJ_{MSE}(W(i-1)) \tag{6}
}
$$